# EDA Master — Hotel No-Show DSS
> 데이터: `data/train.csv` (bookings_weather_pm, 시간 기반 split 후 train 셋)  
> 기간: 2015-07 ~ 2016-12 | 78,703행 × 38컬럼  
> 최종 업데이트: 2026-05-11  

**커버 범위**
1. 타겟 분포 / lead_time / deposit_type / previous_cancellations
2. market_segment & distribution_channel / 날씨 시간 비대칭성 / country / meal / special_requests / ADR
3. Phase 2 방향 근거 — 채널 실효 수익 · 예약 품질 요소 · 음식 낭비 위험
4. City vs Resort 비교 / 상관관계 히트맵 / 요약

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

BLUE   = '#4C9BE8'   # 정상
RED    = '#E85C5C'   # 취소
GRAY   = '#AAAAAA'
GREEN  = '#5BAD72'
ORANGE = '#E8943A'

df = pd.read_csv('../data/train.csv', parse_dates=['arrival_date'])
print(f'shape: {df.shape}')
print(f'취소율: {df["is_canceled"].mean()*100:.1f}%')
df.head(3)

---
## 1. 타겟 분포 — 취소율

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (title, mask) in zip(axes, [
    ('전체', df.index),
    ('City Hotel', df[df['hotel']=='City Hotel'].index),
    ('Resort Hotel', df[df['hotel']=='Resort Hotel'].index),
]):
    sub = df.loc[mask]
    vals = sub['is_canceled'].value_counts(normalize=True).sort_index() * 100
    bars = ax.bar(['정상', '취소'], [vals.get(0,0), vals.get(1,0)],
                  color=[BLUE, RED], width=0.5, edgecolor='white')
    ax.set_title(f'{title}\n(n={len(sub):,})')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylim(0, 85)
    for bar, v in zip(bars, [vals.get(0,0), vals.get(1,0)]):
        ax.text(bar.get_x()+bar.get_width()/2, v+1, f'{v:.1f}%', ha='center', fontweight='bold')

plt.suptitle('취소율 분포', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('City Hotel:  ', df[df['hotel']=='City Hotel']['is_canceled'].mean()*100, '%')
print('Resort Hotel:', df[df['hotel']=='Resort Hotel']['is_canceled'].mean()*100, '%')

---
## 2. lead_time — 핵심 예측 신호
> 취소 예약의 lead_time이 정상 예약보다 훨씬 길다.  
> 막판 예약(≤7일)은 거의 취소 안 한다 — Flexi 슬롯 제외 대상.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 분포 비교
for label, color, name in [(0, BLUE, '정상'), (1, RED, '취소')]:
    axes[0].hist(df[df['is_canceled']==label]['lead_time'].clip(upper=400),
                 bins=50, alpha=0.6, color=color, label=name, density=True)
axes[0].set_title('lead_time 분포 (취소 여부별)')
axes[0].set_xlabel('lead_time (일, 400일 클리핑)')
axes[0].legend()

# 구간별 취소율
bins   = [0, 7, 30, 90, 180, 365, 999]
labels = ['~7일', '8~30일', '31~90일', '91~180일', '181~365일', '365일+']
df['lead_bin'] = pd.cut(df['lead_time'], bins=bins, labels=labels)
rate = df.groupby('lead_bin', observed=True)['is_canceled'].mean() * 100
bars = axes[1].bar(rate.index, rate.values, color=RED, width=0.6, edgecolor='white')
axes[1].set_title('lead_time 구간별 취소율')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].tick_params(axis='x', rotation=30)
for bar, v in zip(bars, rate.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v:.0f}%', ha='center', fontsize=9)

# 평균 비교
means = df.groupby('is_canceled')['lead_time'].mean()
axes[2].bar(['정상', '취소'], means.values, color=[BLUE, RED], width=0.5, edgecolor='white')
axes[2].set_title('평균 lead_time 비교')
axes[2].set_ylabel('일수')
for i, v in enumerate(means.values):
    axes[2].text(i, v+1, f'{v:.0f}일', ha='center', fontweight='bold')

plt.suptitle('lead_time 분석', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. deposit_type — DROP 결정 근거
> Non Refund 취소율 99.2% — 경제적으로 말이 안 됨.  
> B2B 여행사 allotment 계약: release date 이전 블록 반납은 패널티 없음.  
> 타임스탬프 없어 사후 기록 오염 가능성도 배제 불가 → **DROP 확정**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 취소율
rate = df.groupby('deposit_type')['is_canceled'].mean().sort_values(ascending=False) * 100
bars = axes[0].bar(rate.index, rate.values, color=[RED, ORANGE, BLUE][:len(rate)],
                   width=0.5, edgecolor='white')
axes[0].set_title('deposit_type별 취소율')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].tick_params(axis='x', rotation=15)
for bar, v in zip(bars, rate.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+1, f'{v:.1f}%', ha='center', fontweight='bold')

# 건수
counts = df['deposit_type'].value_counts()
axes[1].bar(counts.index, counts.values, color=GRAY, width=0.5, edgecolor='white')
axes[1].set_title('deposit_type 건수')
axes[1].tick_params(axis='x', rotation=15)
for i, (idx, v) in enumerate(counts.items()):
    axes[1].text(i, v+200, f'{v:,}', ha='center', fontsize=9)

# Non Refund의 market_segment 분포
nr = df[df['deposit_type']=='Non Refund']['market_segment'].value_counts()
axes[2].barh(nr.index[:6], nr.values[:6], color=ORANGE, edgecolor='white')
axes[2].set_title('Non Refund — market_segment 분포\n(B2B 패턴 확인)')
for i, v in enumerate(nr.values[:6]):
    axes[2].text(v+10, i, f'{v:,}', va='center', fontsize=9)

plt.suptitle('deposit_type 분석 (DROP 근거)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Non Refund 중 Groups+Offline TA/TO 비율:')
nr_b2b = df[df['deposit_type']=='Non Refund']
b2b_mask = nr_b2b['market_segment'].isin(['Groups', 'Offline TA/TO'])
print(f'  {b2b_mask.sum():,}건 / {len(nr_b2b):,}건 = {b2b_mask.mean()*100:.1f}%')

---
## 4. previous_cancellations — 강한 신호 + B2B 패턴 주의
> 이력 있음 그룹 취소율 91.64% — 압도적 신호.  
> 단, `is_repeated_guest=0`인데 `previous_cancellations≥1`인 행 약 2,674건 (정의 모순).  
> SHAP에서 이 변수 상위 시 'B2B 여행사 패턴'을 잡는 것일 수 있음 → Phase 2 ablation 필요.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 이력 유무별 취소율
df['has_prev'] = (df['previous_cancellations'] > 0).map({True:'있음', False:'없음'})
rate = df.groupby('has_prev')['is_canceled'].mean() * 100
bars = axes[0].bar(rate.index, rate.values, color=[RED, BLUE], width=0.4, edgecolor='white')
axes[0].set_title('이전 취소 이력 유무별 취소율')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, v in zip(bars, rate.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+1, f'{v:.1f}%', ha='center', fontweight='bold')

# 연도별 ≥1 그룹 취소율
yr_rates = []
for yr in sorted(df['arrival_date_year'].unique()):
    sub = df[(df['arrival_date_year']==yr) & (df['previous_cancellations']>=1)]
    if len(sub) > 0:
        yr_rates.append((yr, sub['is_canceled'].mean()*100, len(sub)))
yr_df = pd.DataFrame(yr_rates, columns=['year','rate','n'])
bars2 = axes[1].bar(yr_df['year'].astype(str), yr_df['rate'], color=RED, width=0.4, edgecolor='white')
axes[1].axhline(90, color=GRAY, linestyle='--', label='90% 기준선')
axes[1].set_title('연도별 previous_cancellations≥1 그룹 취소율')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend()
for bar, row in zip(bars2, yr_df.itertuples()):
    axes[1].text(bar.get_x()+bar.get_width()/2, row.rate+0.5,
                 f'{row.rate:.1f}%\n(n={row.n:,})', ha='center', fontsize=8)

# 정의 모순 행 분석 (is_repeated_guest=0 & prev≥1)
mismatch = df[(df['is_repeated_guest']==0) & (df['previous_cancellations']>=1)]
seg_dist = mismatch['market_segment'].value_counts().head(5)
axes[2].barh(seg_dist.index, seg_dist.values, color=ORANGE, edgecolor='white')
axes[2].set_title(f'정의 모순 행 {len(mismatch):,}건\n(is_repeated_guest=0 & prev≥1)\nmarket_segment 분포')
for i, v in enumerate(seg_dist.values):
    axes[2].text(v+5, i, f'{v:,}', va='center', fontsize=9)

plt.suptitle('previous_cancellations 분석', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'정의 모순 행 취소율: {mismatch["is_canceled"].mean()*100:.1f}%')
print(f'정상 정의 행(repeated=1 & prev≥1) 수: {len(df[(df["is_repeated_guest"]==1) & (df["previous_cancellations"]>=1)]):,}')

---
## 5. market_segment & distribution_channel — 채널 패턴
> Phase 2 방향 1 '채널 실효 수익 분석'의 EDA 근거.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col, title in zip(axes,
    ['market_segment', 'distribution_channel'],
    ['market_segment별 취소율', 'distribution_channel별 취소율']):
    grp = df.groupby(col).agg(
        n=('is_canceled','count'),
        cancel_rate=('is_canceled','mean')
    ).sort_values('cancel_rate', ascending=False)
    grp['cancel_rate'] *= 100
    bars = ax.bar(grp.index, grp['cancel_rate'], color=RED, edgecolor='white', width=0.6)
    ax.set_title(title)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.tick_params(axis='x', rotation=25)
    for bar, (_, row) in zip(bars, grp.iterrows()):
        ax.text(bar.get_x()+bar.get_width()/2, row.cancel_rate+0.5,
                f"{row.cancel_rate:.0f}%\n(n={row.n:,})", ha='center', fontsize=8)

plt.suptitle('채널별 취소율', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. 날씨 × lead_time — 시간 비대칭성
> 핵심 발표 슬라이드 근거.  
> lead_time ≤ 30일에서만 강수량 증가 → 취소율 상승 관계가 유의.  
> lead_time > 90일에서는 날씨가 취소율과 무관.  
> 해석: 손님은 도착 임박 시점의 예보를 보고 취소를 결정한다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (subset_mask, title) in zip(axes, [
    (df['lead_time'] <= 30,  'lead_time ≤ 30일 (임박 예약)'),
    (df['lead_time'] > 90,   'lead_time > 90일 (장기 예약)'),
]):
    sub = df[subset_mask]
    try:
        bins = pd.qcut(sub['precipitation_sum'], q=5, duplicates='drop')
        rate = sub.groupby(bins, observed=True)['is_canceled'].mean() * 100
        ax.bar(range(len(rate)), rate.values, color='#5BA4CF', edgecolor='white', width=0.6)
        ax.set_xticks(range(len(rate)))
        ax.set_xticklabels([str(b) for b in rate.index], rotation=30, fontsize=8)
    except Exception:
        ax.text(0.5, 0.5, '데이터 불충분', transform=ax.transAxes, ha='center')
    ax.set_title(title)
    ax.set_xlabel('강수량 구간 (mm)')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    for i, v in enumerate(rate.values if 'rate' in dir() else []):
        ax.text(i, v+0.3, f'{v:.0f}%', ha='center', fontsize=8)

plt.suptitle('강수량 × 취소율 — lead_time 구간별', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# 요약 상관
for mask, label in [(df['lead_time']<=30,'≤30일'), (df['lead_time']>90,'>90일')]:
    corr = df[mask][['precipitation_sum','is_canceled']].corr().iloc[0,1]
    print(f'lead_time {label} — precipitation_sum × is_canceled 상관: {corr:.3f}')

### 6-2. 날씨 변수 분포 (취소 여부별)

In [ ]:
weather_cols = ['precipitation_sum', 'temperature_2m_max', 'temperature_2m_min',
                'wind_speed_10m_max', 'precipitation_hours', 'relative_humidity_2m_mean']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, col in zip(axes.flatten(), weather_cols):
    for label, color, name in [(0,BLUE,'정상'),(1,RED,'취소')]:
        vals = df[df['is_canceled']==label][col].dropna()
        ax.hist(vals, bins=40, alpha=0.55, color=color, label=name, density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.suptitle('날씨 변수 분포 (취소 여부별)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. country — 국적 패턴

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Top15 예약 건수
top15 = df['country'].value_counts().head(15)
axes[0].barh(top15.index[::-1], top15.values[::-1], color=BLUE, edgecolor='white')
axes[0].set_title('국적별 예약 건수 Top 15')
for i, v in enumerate(top15.values[::-1]):
    axes[0].text(v+100, i, f'{v:,}', va='center', fontsize=8)

# Top15 국가 취소율
top15_countries = df['country'].value_counts().head(15).index
rate = (df[df['country'].isin(top15_countries)]
        .groupby('country')['is_canceled'].mean()
        .loc[top15_countries] * 100)
colors = [RED if v > 40 else ORANGE if v > 25 else BLUE for v in rate.values]
axes[1].barh(rate.index[::-1], rate.values[::-1], color=colors[::-1], edgecolor='white')
axes[1].axvline(df['is_canceled'].mean()*100, color=GRAY, linestyle='--', label='전체 평균')
axes[1].set_title('국적별 취소율 (Top 15)')
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend()

plt.suptitle('국적(country) 분석', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. meal — 식사 플랜 & 음식 낭비 연결
> Phase 2 방향 3 '음식 낭비 예측'의 EDA 근거.  
> HB/FB 예약이 취소되면 이미 발주된 식재료가 낭비된다.  
> EU CSRD 2024 — 음식 낭비가 ESG 보고 항목.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 식사 플랜별 건수
meal_counts = df['meal'].value_counts()
axes[0].bar(meal_counts.index, meal_counts.values, color=BLUE, edgecolor='white', width=0.5)
axes[0].set_title('meal 플랜별 건수')
for i, (idx, v) in enumerate(meal_counts.items()):
    axes[0].text(i, v+200, f'{v:,}', ha='center', fontsize=9)

# 식사 플랜별 취소율
rate = df.groupby('meal')['is_canceled'].mean().sort_values(ascending=False) * 100
colors = [RED if m in ['HB','FB'] else ORANGE if m=='BB' else BLUE for m in rate.index]
bars = axes[1].bar(rate.index, rate.values, color=colors, edgecolor='white', width=0.5)
axes[1].set_title('meal 플랜별 취소율\n(HB/FB = 음식 낭비 고위험)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, v in zip(bars, rate.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v:.1f}%', ha='center', fontweight='bold')

# HB/FB 예약 중 취소 건수 (음식 낭비 잠재량)
food_risk = df[df['meal'].isin(['HB','FB'])]
food_canceled = food_risk['is_canceled'].sum()
food_total    = len(food_risk)
axes[2].pie([food_canceled, food_total-food_canceled],
            labels=[f'취소\n({food_canceled:,}건)', f'정상\n({food_total-food_canceled:,}건)'],
            colors=[RED, BLUE], autopct='%1.1f%%', startangle=90)
axes[2].set_title(f'HB+FB 예약 취소 비율\n(총 {food_total:,}건 — 음식 낭비 잠재량)')

plt.suptitle('meal 분석', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. total_of_special_requests — 예약 몰입도
> Phase 2 방향 2 'Booking Quality Score'의 핵심 근거.  
> special_requests가 많을수록 취소율이 낮다.  
> 해석: 이 호텔에 와야 할 이유가 있는 손님이 요청도 많이 한다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 요청 수별 취소율
rate = df.groupby('total_of_special_requests')['is_canceled'].mean() * 100
colors = [RED if i==0 else ORANGE if i<=1 else BLUE for i in rate.index]
bars = axes[0].bar(rate.index.astype(str), rate.values, color=colors, edgecolor='white', width=0.6)
axes[0].set_title('total_of_special_requests별 취소율')
axes[0].set_xlabel('특별 요청 수')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, v in zip(bars, rate.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v:.1f}%', ha='center', fontsize=9)

# 분포 비교 (취소 vs 정상)
for label, color, name in [(0,BLUE,'정상'),(1,RED,'취소')]:
    vals = df[df['is_canceled']==label]['total_of_special_requests'].value_counts(normalize=True).sort_index() * 100
    axes[1].plot(vals.index, vals.values, color=color, marker='o', label=name, linewidth=2)
axes[1].set_title('special_requests 분포 비교 (정상 vs 취소)')
axes[1].set_xlabel('특별 요청 수')
axes[1].set_ylabel('비율 (%)')
axes[1].legend()

plt.suptitle('total_of_special_requests — 예약 몰입도 신호', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('요청 수 = 0인 예약 취소율:', df[df['total_of_special_requests']==0]['is_canceled'].mean()*100, '%')
print('요청 수 ≥ 2인 예약 취소율:', df[df['total_of_special_requests']>=2]['is_canceled'].mean()*100, '%')

---
## 10. ADR & 체류 기간 — 예약 가치 신호

In [ ]:
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ADR 분포
for label, color, name in [(0,BLUE,'정상'),(1,RED,'취소')]:
    axes[0].hist(df[(df['is_canceled']==label) & (df['adr']>0) & (df['adr']<500)]['adr'],
                 bins=50, alpha=0.6, color=color, label=name, density=True)
axes[0].set_title('ADR 분포 (취소 여부별)')
axes[0].set_xlabel('ADR (€)')
axes[0].legend()

# 체류 기간별 취소율
night_bins = [0,1,3,7,14,999]
night_labels = ['0박','1~3박','4~7박','8~14박','15박+']
df['night_bin'] = pd.cut(df['total_nights'], bins=night_bins, labels=night_labels)
rate = df.groupby('night_bin', observed=True)['is_canceled'].mean() * 100
bars = axes[1].bar(rate.index, rate.values, color=RED, width=0.6, edgecolor='white')
axes[1].set_title('체류 기간별 취소율')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, v in zip(bars, rate.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v:.0f}%', ha='center', fontsize=9)

# ADR × 취소 여부 박스플롯
plot_df = df[(df['adr']>0) & (df['adr']<500)]
axes[2].boxplot(
    [plot_df[plot_df['is_canceled']==0]['adr'].values,
     plot_df[plot_df['is_canceled']==1]['adr'].values],
    labels=['정상','취소'],
    patch_artist=True,
    boxprops=dict(facecolor=BLUE, alpha=0.6)
)
axes[2].set_title('ADR 박스플롯 (취소 여부별)')
axes[2].set_ylabel('ADR (€)')

plt.suptitle('ADR & 체류 기간 분석', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'평균 ADR — 정상: €{df[df["is_canceled"]==0]["adr"].mean():.1f}  취소: €{df[df["is_canceled"]==1]["adr"].mean():.1f}')

---
## 11. 채널 실효 수익 분석 (Phase 2 방향 1)
> `Effective ADR = ADR × (1 - cancel_rate)`  
> 호텔이 체감하는 실제 채널별 수익성.

In [ ]:
# 채널별 실효 ADR
channel_stats = df.groupby('distribution_channel').agg(
    n=('is_canceled','count'),
    avg_adr=('adr','mean'),
    cancel_rate=('is_canceled','mean')
).assign(
    effective_adr=lambda x: x['avg_adr'] * (1 - x['cancel_rate'])
).sort_values('effective_adr', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

x = range(len(channel_stats))
width = 0.35
bars1 = axes[0].bar([i-width/2 for i in x], channel_stats['avg_adr'],
                    width=width, label='액면 ADR', color=BLUE, edgecolor='white')
bars2 = axes[0].bar([i+width/2 for i in x], channel_stats['effective_adr'],
                    width=width, label='실효 ADR', color=GREEN, edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(channel_stats.index, rotation=20)
axes[0].set_title('채널별 액면 ADR vs 실효 ADR (취소 반영)')
axes[0].set_ylabel('ADR (€)')
axes[0].legend()
for bar in bars1:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'€{bar.get_height():.0f}', ha='center', fontsize=8)
for bar in bars2:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'€{bar.get_height():.0f}', ha='center', fontsize=8, color='darkgreen')

# market_segment 실효 ADR
seg_stats = df.groupby('market_segment').agg(
    n=('is_canceled','count'),
    avg_adr=('adr','mean'),
    cancel_rate=('is_canceled','mean')
).assign(
    effective_adr=lambda x: x['avg_adr'] * (1 - x['cancel_rate'])
).sort_values('effective_adr', ascending=True)

colors_seg = [RED if v < 50 else ORANGE if v < 70 else GREEN for v in seg_stats['effective_adr']]
axes[1].barh(seg_stats.index, seg_stats['effective_adr'], color=colors_seg, edgecolor='white')
axes[1].set_title('market_segment별 실효 ADR')
axes[1].set_xlabel('실효 ADR (€)')
for i, v in enumerate(seg_stats['effective_adr']):
    axes[1].text(v+0.5, i, f'€{v:.0f}', va='center', fontsize=9)

plt.suptitle('채널 실효 수익 분석 — Phase 2 방향 1', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n채널별 실효 ADR 상세:')
print(channel_stats[['n','avg_adr','cancel_rate','effective_adr']].to_string())

### 11-2. 채널 × lead_time 구간 실효 수익 매트릭스

In [ ]:
lead_bins   = [0, 14, 30, 90, 180, 999]
lead_labels = ['0-14일', '15-30일', '31-90일', '91-180일', '181일+']
df['lead_bucket'] = pd.cut(df['lead_time'], bins=lead_bins, labels=lead_labels)

matrix = df.groupby(['distribution_channel','lead_bucket'], observed=True).agg(
    n=('is_canceled','count'),
    cancel_rate=('is_canceled','mean'),
    avg_adr=('adr','mean')
).assign(
    effective_adr=lambda x: x['avg_adr'] * (1 - x['cancel_rate'])
).reset_index()

pivot_cancel = matrix.pivot(index='distribution_channel', columns='lead_bucket', values='cancel_rate') * 100
pivot_yield  = matrix.pivot(index='distribution_channel', columns='lead_bucket', values='effective_adr')

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

sns.heatmap(pivot_cancel, annot=True, fmt='.0f', cmap='RdYlGn_r',
            ax=axes[0], linewidths=0.5, annot_kws={'size':10})
axes[0].set_title('채널 × lead_time 구간별 취소율 (%)')

sns.heatmap(pivot_yield, annot=True, fmt='.0f', cmap='RdYlGn',
            ax=axes[1], linewidths=0.5, annot_kws={'size':10})
axes[1].set_title('채널 × lead_time 구간별 실효 ADR (€)')

plt.suptitle('채널 × 리드타임 매트릭스', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 12. 예약 품질 요소 분석 (Phase 2 방향 2)
> Booking Quality Score (BQS) 설계 근거.  
> 취소 확률이 낮더라도 ADR 낮고 체류 짧고 요청 없으면 품질 낮은 예약.  
> BQS = ADR + 체류기간 + special_requests + channel + repeated_guest + cancel_proba

In [ ]:
# 재방문 여부별 패턴
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 재방문 × 취소율
rate = df.groupby('is_repeated_guest')['is_canceled'].mean() * 100
axes[0,0].bar(['신규','재방문'], rate.values, color=[ORANGE, GREEN], width=0.4, edgecolor='white')
axes[0,0].set_title('재방문 여부별 취소율')
axes[0,0].yaxis.set_major_formatter(mtick.PercentFormatter())
for i, v in enumerate(rate.values):
    axes[0,0].text(i, v+0.5, f'{v:.1f}%', ha='center', fontweight='bold')

# customer_type × 취소율
rate2 = df.groupby('customer_type')['is_canceled'].mean().sort_values(ascending=False) * 100
bars = axes[0,1].bar(rate2.index, rate2.values, color=RED, width=0.5, edgecolor='white')
axes[0,1].set_title('customer_type별 취소율')
axes[0,1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0,1].tick_params(axis='x', rotation=20)
for bar, v in zip(bars, rate2.values):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v:.1f}%', ha='center', fontsize=9)

# special_requests × ADR 산점도 (샘플)
sample = df[(df['adr']>0) & (df['adr']<500)].sample(3000, random_state=42)
colors_scatter = [RED if c==1 else BLUE for c in sample['is_canceled']]
axes[0,2].scatter(sample['total_of_special_requests'], sample['adr'],
                  c=colors_scatter, alpha=0.3, s=10)
axes[0,2].set_title('special_requests × ADR\n(파랑=정상, 빨강=취소)')
axes[0,2].set_xlabel('special_requests')
axes[0,2].set_ylabel('ADR (€)')

# 품질 세그먼트 비교 (고품질 vs 저품질)
high_q = df[
    (df['total_of_special_requests'] >= 2) &
    (df['distribution_channel'] == 'Direct') &
    (df['total_nights'] >= 3)
]
low_q = df[
    (df['total_of_special_requests'] == 0) &
    (df['distribution_channel'].isin(['TA/TO'])) &
    (df['lead_time'] > 90)
]

for ax, grp, title, color in [
    (axes[1,0], high_q, f'고품질 세그먼트\n(n={len(high_q):,})', GREEN),
    (axes[1,1], low_q,  f'저품질 세그먼트\n(n={len(low_q):,})', RED),
]:
    cancel_rate = grp['is_canceled'].mean() * 100
    ax.bar(['취소율'], [cancel_rate], color=color, width=0.3)
    ax.set_title(title)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_ylim(0, 100)
    ax.text(0, cancel_rate+1, f'{cancel_rate:.1f}%', ha='center', fontweight='bold', fontsize=14)
    avg_adr = grp[grp['adr']>0]['adr'].mean()
    ax.set_xlabel(f'평균 ADR: €{avg_adr:.0f}')

# BQS 구성 요소 상관
bqs_cols = ['is_canceled','lead_time','total_of_special_requests','adr','total_nights','is_repeated_guest']
corr_bqs = df[bqs_cols].corr()
sns.heatmap(corr_bqs, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[1,2], linewidths=0.4, annot_kws={'size':9})
axes[1,2].set_title('BQS 구성 요소 상관관계')

plt.suptitle('예약 품질 요소 분석 — Phase 2 방향 2', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 13. 음식 낭비 위험 분석 (Phase 2 방향 3)
> meal × 취소 위험 조합으로 예상 낭비 규모 추정.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# meal × 취소율 히트맵 (hotel별)
meal_hotel = df.groupby(['hotel','meal'])['is_canceled'].mean().unstack() * 100
sns.heatmap(meal_hotel, annot=True, fmt='.1f', cmap='RdYlGn_r',
            ax=axes[0], linewidths=0.5, annot_kws={'size':10})
axes[0].set_title('호텔별 × meal별 취소율 (%)')

# 월별 HB/FB 고위험 예약 수 (계절성)
food_risk_monthly = (df[df['meal'].isin(['HB','FB'])]
                     .groupby('arrival_date_month')['is_canceled']
                     .agg(['sum','count'])
                     .rename(columns={'sum':'canceled','count':'total'}))
food_risk_monthly['waste_risk_rate'] = food_risk_monthly['canceled'] / food_risk_monthly['total'] * 100
axes[1].bar(food_risk_monthly.index, food_risk_monthly['canceled'],
            color=RED, edgecolor='white', label='HB/FB 취소 건수')
ax2 = axes[1].twinx()
ax2.plot(food_risk_monthly.index, food_risk_monthly['waste_risk_rate'],
         color=ORANGE, marker='o', linewidth=2, label='취소율')
axes[1].set_title('월별 HB/FB 취소 건수 & 취소율')
axes[1].set_xlabel('월')
axes[1].set_ylabel('취소 건수')
ax2.set_ylabel('취소율 (%)')
axes[1].legend(loc='upper left')
ax2.legend(loc='upper right')

# lead_time별 HB/FB 취소율 (언제 알 수 있는가)
food_df = df[df['meal'].isin(['HB','FB'])].copy()
food_df['lead_bucket'] = pd.cut(food_df['lead_time'], bins=lead_bins, labels=lead_labels)
rate_food = food_df.groupby('lead_bucket', observed=True)['is_canceled'].mean() * 100
bars = axes[2].bar(rate_food.index, rate_food.values, color=RED, width=0.6, edgecolor='white')
axes[2].set_title('HB/FB 예약\nlead_time 구간별 취소율')
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[2].tick_params(axis='x', rotation=25)
for bar, v in zip(bars, rate_food.values):
    axes[2].text(bar.get_x()+bar.get_width()/2, v+0.5, f'{v:.0f}%', ha='center', fontsize=9)

plt.suptitle('음식 낭비 위험 분석 — Phase 2 방향 3', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

hb_fb = df[df['meal'].isin(['HB','FB'])]
print(f'HB/FB 전체 건수: {len(hb_fb):,}')
print(f'HB/FB 취소 건수: {hb_fb["is_canceled"].sum():,} ({hb_fb["is_canceled"].mean()*100:.1f}%)')
print(f'→ 발주 조정이 없으면 이만큼이 낭비될 가능성')

---
## 14. 상관관계 히트맵 — 수치형 변수 전체

In [ ]:
num_cols = ['is_canceled', 'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights',
            'adults', 'previous_cancellations', 'booking_changes',
            'adr', 'total_of_special_requests', 'is_repeated_guest',
            'required_car_parking_spaces',
            'precipitation_sum', 'temperature_2m_max', 'temperature_2m_min',
            'wind_speed_10m_max', 'precipitation_hours',
            'relative_humidity_2m_mean', 'cloud_cover_mean']

# 존재하는 컬럼만
num_cols = [c for c in num_cols if c in df.columns]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-0.6, vmax=0.6, ax=ax,
            linewidths=0.4, annot_kws={'size': 7})
ax.set_title('수치형 변수 상관관계 히트맵', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

top_corr = corr['is_canceled'].drop('is_canceled').abs().sort_values(ascending=False)
print('취소율과 상관 높은 변수 Top 8:')
print(top_corr.head(8).to_string())

---
## 15. City vs Resort 비교 — 자연실험
> 같은 분석이 두 호텔에서 어떻게 다르게 나오는가.  
> 동일 모델이 서로 다른 시장 구조에서 다른 신호를 내는지 확인.

In [ ]:
city   = df[df['hotel']=='City Hotel']
resort = df[df['hotel']=='Resort Hotel']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# lead_time 구간별 취소율 비교
for ax, sub, title, color in [
    (axes[0,0], city,   'City Hotel', BLUE),
    (axes[0,1], resort, 'Resort Hotel', ORANGE),
]:
    sub2 = sub.copy()
    sub2['lead_bin'] = pd.cut(sub2['lead_time'], bins=bins, labels=labels)
    rate = sub2.groupby('lead_bin', observed=True)['is_canceled'].mean() * 100
    ax.bar(rate.index, rate.values, color=color, width=0.6, edgecolor='white')
    ax.set_title(f'{title}\nlead_time별 취소율')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.tick_params(axis='x', rotation=30)

# 월별 취소율 비교
for sub, color, name in [(city,BLUE,'City'),(resort,ORANGE,'Resort')]:
    monthly = sub.groupby('arrival_date_month')['is_canceled'].mean() * 100
    axes[0,2].plot(monthly.index, monthly.values, color=color, marker='o', label=name, linewidth=2)
axes[0,2].set_title('월별 취소율 비교')
axes[0,2].set_xlabel('월')
axes[0,2].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0,2].legend()

# market_segment 믹스 비교
for ax, sub, title in [
    (axes[1,0], city,   'City Hotel — market_segment'),
    (axes[1,1], resort, 'Resort Hotel — market_segment'),
]:
    seg = sub['market_segment'].value_counts(normalize=True) * 100
    ax.barh(seg.index[:6], seg.values[:6], color=BLUE, edgecolor='white')
    ax.set_title(title)
    ax.set_xlabel('%')

# ADR 분포 비교
for sub, color, name in [(city,BLUE,'City'),(resort,ORANGE,'Resort')]:
    vals = sub[(sub['adr']>0)&(sub['adr']<500)]['adr']
    axes[1,2].hist(vals, bins=50, alpha=0.6, color=color, label=name, density=True)
axes[1,2].set_title('ADR 분포 비교')
axes[1,2].set_xlabel('ADR (€)')
axes[1,2].legend()

plt.suptitle('City Hotel vs Resort Hotel 비교 — 자연실험', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    'City Hotel': [
        f"{city['is_canceled'].mean()*100:.1f}%",
        f"{city['lead_time'].mean():.0f}일",
        f"€{city[city['adr']>0]['adr'].mean():.1f}",
        f"{city['total_nights'].mean():.1f}박",
    ],
    'Resort Hotel': [
        f"{resort['is_canceled'].mean()*100:.1f}%",
        f"{resort['lead_time'].mean():.0f}일",
        f"€{resort[resort['adr']>0]['adr'].mean():.1f}",
        f"{resort['total_nights'].mean():.1f}박",
    ]
}, index=['취소율','평균 lead_time','평균 ADR','평균 체류기간'])
print(summary)

---
## 16. 요약 인사이트

| 분석 | 핵심 발견 | 모델/발표 활용 |
|------|----------|---------------|
| 취소율 분포 | 전체 36%, City 41.7% vs Resort 27.8% | 두 호텔 분리 분석 필요 |
| lead_time | 취소 예약 lead_time >> 정상 / ~7일 이내 거의 안 취소 | SHAP 상위 예상 신호 |
| deposit_type | Non Refund 99.2% 취소 = B2B allotment 패턴 | **DROP 확정** |
| previous_cancellations | ≥1 그룹 91.64% 취소 / 정의 모순 2,674건 | B2B 해석 주의, Phase 2 ablation |
| market_segment | Groups/Online TA 취소율 높음 / Corporate/Direct 낮음 | 채널 실효 수익 분석 근거 |
| 날씨 × lead_time | ≤30일에서만 강수량↑→취소율↑ / >90일은 무관 | **발표 핵심 슬라이드** |
| special_requests | 0개 취소율 높음 / ≥2개 낮음 = 예약 몰입도 신호 | Booking Quality Score 근거 |
| meal HB/FB | 취소율 높고 발주된 식재료 낭비 위험 | 음식 낭비 예측 방향 |
| 채널 실효 수익 | Direct > Corporate > TA/TO (취소 반영 후) | Phase 2 방향 1 |
| City vs Resort | lead_time 패턴, 시즌성, ADR 모두 다름 | 자연실험 비교 분석 |
